In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")  # loaded but will be skipped in aggregation step

# Parse timestamps to UTC datetime (rule 10)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Normalize column names and add source_file
metric = metric.assign(source_file='metric', kpi_name=metric['kpi_name'])
trace = trace.assign(source_file='trace', kpi_name=trace['trace_name'])
log = log.assign(source_file='log', kpi_name=log['log_name'])

# Select only needed columns for aggregation
metric_cols = metric[['source_file', 'cmdb_id', 'kpi_name', 'value']]
trace_cols = trace[['source_file', 'cmdb_id', 'kpi_name', 'value']]
log_cols = log[['source_file', 'cmdb_id', 'kpi_name', 'value']]

# Define aggregation function including global P95
agg_funcs = {
    'value': [
        ('count', 'count'),
        ('mean', 'mean'),
        ('std', 'std'),
        ('p95', lambda x: x.quantile(0.95))
    ]
}

# Helper to perform groupby and flatten MultiIndex columns
def aggregate_df(df):
    g = df.groupby(['source_file', 'cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        std='std',
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    # Ensure column order
    return g[['source_file', 'cmdb_id', 'kpi_name', 'count', 'mean', 'std', 'p95']]

metric_agg = aggregate_df(metric_cols)
trace_agg = aggregate_df(trace_cols)
log_agg = aggregate_df(log_cols)

# Combine aggregated tables
combined = pd.concat([metric_agg, trace_agg, log_agg], ignore_index=True)

# Sort by p95 descending and limit to top 50
combined_top50 = combined.sort_values('p95', ascending=False).head(50).reset_index(drop=True)

# Summary: distinct cmdb_id and distinct KPI names per source_file
summary_counts = combined.groupby('source_file').agg(
    distinct_cmdb_id=('cmdb_id', 'nunique'),
    distinct_kpi_name=('kpi_name', 'nunique')
).reset_index()

# Display results (use variable names as per rules)
combined_top50, summary_counts

```
Out[1]:
```
Summary of results (plain English):

- The top p95 values are dominated by metric KPIs (mostly "mem"). The highest p95 is for metric / ts-travel-service / mem: about 1.001×10^9 (count=25). The next highest p95s include:
  - metric / ts-voucher-mysql / mem: ≈ 6.0048×10^8 (count=25)
  - metric / ts-food-mysql / mem: ≈ 6.0004×10^8 (count=25)
  - metric / ts-travel2-service / mem: ≈ 2.7807×10^8 (count=25)
  - metric / ts-station-service / mem: ≈ 2.6804×10^8 (count=25)

- The combined table returned the top 50 KPI groups sorted by p95 (each row shows source_file, cmdb_id, kpi_name, count, mean, std, p95). Many of the top rows are memory-related metric KPIs.

- Small summary of distinct counts by source_file:
  - log: 45 distinct cmdb_id, 2 distinct KPI names
  - metric: 68 distinct cmdb_id, 8 distinct KPI names
  - trace: 28 distinct cmdb_id, 220 distinct KPI names

Notes:
- The table shown was limited to the top 50 rows by p95. If you want more rows or a specific subset (e.g., traces or logs only), I can show that.

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file               cmdb_id kpi_name  count          mean           std           p95
0       metric     ts-travel-service      mem     25  5.749493e+08  2.978027e+08  1.001282e+09
1       metric      ts-voucher-mysql      mem     25  6.004818e+08  0.000000e+00  6.004818e+08
2       metric         ts-food-mysql      mem     25  6.000394e+08  0.000000e+00  6.000394e+08
3       metric    ts-travel2-service      mem     25  2.776743e+08  2.251969e+05  2.780691e+08
4       metric    ts-station-service      mem     25  2.677028e+08  3.120229e+05  2.680370e+08
..         ...                   ...      ...    ...           ...           ...           ...
45      metric      ts-payment-mongo      mem     25  7.111590e+07  1.032767e+05  7.123116e+07
46      metric        ts-price-mongo      mem     25  7.105217e+07  6.423370e+04  7.112240e+07
47      metric         ts-user-mongo      mem     25  7.069883e+07  3.901595e+04  7.075373e+07
48      metric  ts-order-other-mongo      mem     25  7.004328e+07  3.405767e+05  7.074393e+07
49      metric     ts-security-mongo      mem     25  7.051664e+07  1.127616e+05  7.060455e+07

[50 rows x 7 columns],   source_file  distinct_cmdb_id  distinct_kpi_name
0         log                45                  2
1      metric                68                  8
2       trace                28                220)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric if available, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed and kpi_name exists
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
if 'kpi_name' not in metric.columns and 'kpi' in metric.columns:
    metric = metric.rename(columns={'kpi': 'kpi_name'})

# Target services
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict metric to those services (for global p95 calculation and later)
metric_services = metric[metric['cmdb_id'].isin(services)].copy()

# Compute global P95 per (cmdb_id, kpi_name) using the full available series (no time filter)
global_p95_df = metric_services.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
global_p95_df = global_p95_df.rename(columns={'value': 'global_p95'})

# Total number of (cmdb_id, kpi_name) groups examined
total_groups = global_p95_df.shape[0]

# Define incident window (UTC)
start = pd.to_datetime('2024-01-25 16:45:00', utc=True)
end = pd.to_datetime('2024-01-25 17:15:00', utc=True)

# Filter metric to the incident window
window_df = metric_services[(metric_services['timestamp'] >= start) & (metric_services['timestamp'] <= end)].copy()

if window_df.empty:
    # Empty result structure
    final_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ])
else:
    # Merge global p95 into the windowed data
    window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')

    # Flag anomalies where value >= global_p95
    window_with_p95['is_anomaly'] = window_with_p95['value'] >= window_with_p95['global_p95']

    # Aggregate counts, anomaly counts, max, and keep global_p95
    grp = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False)
    agg_counts = grp['value'].count().rename(columns={'value':'count_in_window'})
    agg_anoms = grp['is_anomaly'].sum().rename(columns={'is_anomaly':'count_anomalies_in_window'})
    agg_max = grp['value'].max().rename(columns={'value':'max_value_in_window'})
    agg_p95 = grp['global_p95'].first().rename(columns={'global_p95':'global_p95'})

    # Merge aggregations
    final_df = agg_counts.merge(agg_anoms, on=['cmdb_id','kpi_name'])
    final_df = final_df.merge(agg_max, on=['cmdb_id','kpi_name'])
    final_df = final_df.merge(agg_p95, on=['cmdb_id','kpi_name'])

    # Earliest anomaly timestamp per group (if any)
    anomalies = window_with_p95[window_with_p95['is_anomaly']]
    if not anomalies.empty:
        earliest = anomalies.groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp'].min()
        earliest = earliest.rename(columns={'timestamp':'earliest_anomaly_timestamp'})
        final_df = final_df.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    else:
        final_df['earliest_anomaly_timestamp'] = pd.NaT

    # Convert earliest_anomaly_timestamp to integer seconds or NaN
    def to_int_seconds(ts):
        if pd.isna(ts):
            return np.nan
        return int(ts.timestamp())
    final_df['earliest_anomaly_timestamp'] = final_df['earliest_anomaly_timestamp'].apply(to_int_seconds)

    # Ensure integer counts
    final_df['count_in_window'] = final_df['count_in_window'].astype(int)
    final_df['count_anomalies_in_window'] = final_df['count_anomalies_in_window'].astype(int)

    # Select and order columns
    final_df = final_df[[
        'cmdb_id','kpi_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ]]

# Sort by count_anomalies_in_window desc then global_p95 desc, limit to 20
final_df = final_df.sort_values(by=['count_anomalies_in_window','global_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# Return compact results: final_df and total_groups
final_df, total_groups

```
Out[2]:
```
Summary (plain English):

- I examined 35 (cmdb_id, kpi_name) groups across the five services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.

- In the incident window (2024-01-25 16:45:00–17:15:00 UTC) the top anomalous groups are:
  1. ts-route-service / socket — 3 anomaly points (value >= global P95). earliest_anomaly_timestamp: 1706201400, max_value_in_window: 13.4, global_p95: 13.0.
  2. ts-travel-service / mem — 2 anomaly points. earliest_anomaly_timestamp: 1706202600, max_value_in_window: 1.073742e+09, global_p95: 1.001282e+09.
  3. ts-order-service / mem — 2 anomaly points. earliest_anomaly_timestamp: 1706202600, max_value_in_window: 2.651251e+08, global_p95: 2.650549e+08.
  4. ts-route-service / mem — 2 anomaly points. earliest_anomaly_timestamp: 1706202660, max_value_in_window: 2.500041e+08, global_p95: 2.500015e+08.
  5. ts-train-service / mem — 2 anomaly points. earliest_anomaly_timestamp: 1706202660, max_value_in_window: 2.477261e+08, global_p95: 2.477043e+08.

- Additional notable patterns in the top results: multiple services show 2 anomaly points for mem, diskio, cpu, socket, and workload KPIs (see the returned table for details and the earliest anomaly timestamps in epoch seconds).

- Overall takeaway: ts-route-service.socket had the most anomalies (3) in the incident window; several memory and disk I/O KPIs across the services had values exceeding their global P95 (mostly 2 anomaly points each).

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  kpi_name  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp  max_value_in_window    global_p95
0    ts-route-service    socket               25                          3                  1706201400         1.340000e+01  1.300000e+01
1   ts-travel-service       mem               25                          2                  1706202600         1.073742e+09  1.001282e+09
2    ts-order-service       mem               25                          2                  1706202600         2.651251e+08  2.650549e+08
3    ts-route-service       mem               25                          2                  1706202660         2.500041e+08  2.500015e+08
4    ts-train-service       mem               25                          2                  1706202660         2.477261e+08  2.477043e+08
5     ts-auth-service       mem               25                          2                  1706202480         2.422106e+08  2.407865e+08
6   ts-travel-service    diskio               25                          2                  1706202180         2.730113e+07  2.695014e+07
7    ts-train-service    diskio               25                          2                  1706202240         1.393841e+06  1.326914e+06
8    ts-route-service    diskio               25                          2                  1706202240         1.476152e+06  1.303356e+06
9    ts-order-service    diskio               25                          2                  1706202240         1.351331e+06  1.237331e+06
10    ts-auth-service    diskio               25                          2                  1706201580         6.006607e+04  5.657595e+04
11  ts-travel-service       cpu               25                          2                  1706202660         4.045871e+01  3.936146e+01
12   ts-route-service  workload               25                          2                  1706201700         3.322660e+01  3.310464e+01
13  ts-travel-service    socket               25                          2                  1706202300         2.300000e+01  2.265000e+01
14   ts-train-service  workload               25                          2                  1706202240         2.160000e+01  2.123398e+01
15   ts-train-service    socket               25                          2                  1706202120         1.900000e+01  1.864667e+01
16    ts-auth-service    socket               25                          2                  1706201520         1.760000e+01  1.754333e+01
17    ts-auth-service       cpu               25                          2                  1706201520         1.673160e+01  1.583317e+01
18   ts-order-service    socket               25                          2                  1706201880         1.511667e+01  1.501333e+01
19  ts-travel-service  workload               25                          2                  1706201700         1.093300e+01  1.023063e+01, 35)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace if available, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed and trace_name exists
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
if 'trace_name' not in trace.columns and 'trace' in trace.columns:
    trace = trace.rename(columns={'trace': 'trace_name'})

# Target services (reuse from previous cell if available)
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to the five services
trace_services = trace[trace['cmdb_id'].isin(services)].copy()

# Compute global P95 per (cmdb_id, trace_name) using the full available series (no time filter)
global_p95_df = trace_services.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95)
global_p95_df = global_p95_df.rename(columns={'value': 'global_p95'})

# Incident window (UTC)
start = pd.to_datetime('2024-01-25 16:45:00', utc=True)
end = pd.to_datetime('2024-01-25 17:15:00', utc=True)

# Filter to the incident window
window_df = trace_services[(trace_services['timestamp'] >= start) & (trace_services['timestamp'] <= end)].copy()

if window_df.empty:
    final_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ])
else:
    # Merge global p95 into the windowed data
    window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id','trace_name'], how='left')

    # Flag anomalies where value >= global_p95
    window_with_p95['is_anomaly'] = window_with_p95['value'] >= window_with_p95['global_p95']

    # Aggregate metrics per group
    grp = window_with_p95.groupby(['cmdb_id','trace_name'], as_index=False)
    agg_counts = grp['value'].count().rename(columns={'value':'count_in_window'})
    agg_anoms = grp['is_anomaly'].sum().rename(columns={'is_anomaly':'count_anomalies_in_window'})
    agg_max = grp['value'].max().rename(columns={'value':'max_value_in_window'})
    agg_p95 = grp['global_p95'].first().rename(columns={'global_p95':'global_p95'})

    final_df = agg_counts.merge(agg_anoms, on=['cmdb_id','trace_name'])
    final_df = final_df.merge(agg_max, on=['cmdb_id','trace_name'])
    final_df = final_df.merge(agg_p95, on=['cmdb_id','trace_name'])

    # Earliest anomaly timestamp per group (if any)
    anomalies = window_with_p95[window_with_p95['is_anomaly']]
    if not anomalies.empty:
        earliest = anomalies.groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min()
        earliest = earliest.rename(columns={'timestamp':'earliest_anomaly_timestamp'})
        # convert to integer seconds
        earliest['earliest_anomaly_timestamp'] = earliest['earliest_anomaly_timestamp'].apply(lambda ts: int(ts.timestamp()))
        final_df = final_df.merge(earliest, on=['cmdb_id','trace_name'], how='left')
    else:
        final_df['earliest_anomaly_timestamp'] = np.nan

    # Ensure integer counts
    final_df['count_in_window'] = final_df['count_in_window'].astype(int)
    final_df['count_anomalies_in_window'] = final_df['count_anomalies_in_window'].astype(int)

    # Order columns
    final_df = final_df[[
        'cmdb_id','trace_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ]]

# Sort by anomaly count desc then global_p95 desc, limit to 20 rows
final_df = final_df.sort_values(by=['count_anomalies_in_window','global_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# Distinct trace_name values encountered for these services (limit to top 50)
distinct_trace_names = pd.DataFrame({'trace_name': sorted(trace_services['trace_name'].unique())[:50]})

# Output compact results
final_df, distinct_trace_names

```
Out[3]:
```
Summary (plain English):

- I examined trace telemetry for the five services in the incident window (2024-01-25 16:45–17:15 UTC). The top results are dominated by trace error_rate metrics for many services.

- Key finding: multiple error_rate trace metrics (e.g., trace.from_root.error_rate, trace.from_ts-auth-service.error_rate, trace.to_ts-auth-service.error_rate, trace.from_root.error_rate for ts-order-service, etc.) each have 24 points in the window and were all flagged as anomalies (count_anomalies_in_window = 24). For these groups:
  - max_value_in_window = 0.0 and global_p95 = 0.0, so every value (0.0) met the condition value >= global_p95 and was counted as an anomaly.
  - earliest_anomaly_timestamp reported as 1706201280 (epoch seconds), which lies inside the incident window.

- Distinct trace_name types encountered (sample / top entries include): trace.from_root.duration_mean, trace.from_root.duration_p95, trace.from_root.error_rate, trace.from_root.row_count, trace.from_ts-...duration_mean/duration_p95/error_rate, trace.to_ts-...error_rate, and many service-specific duration_p95/error_rate/row_count variants (the trace_name list contains many entries; up to 50 were returned).

- Interpretation / recommendation:
  - The anomaly count for error_rate is misleading here because global_p95 = 0.0. When the baseline P95 is zero, using value >= global_p95 marks zeros as anomalies. Treat zero-threshold metrics specially (e.g., require value > global_p95, require global_p95 > 0, or use a different anomaly rule).
  - Next steps: inspect non-error_rate trace metrics (duration_p95, duration_mean) and other telemetry (metrics/logs) to find meaningful deviations, and adjust anomaly detection logic for zero-baseline metrics.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                       trace_name  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp  max_value_in_window  global_p95
0    ts-auth-service                       trace.from_root.error_rate               24                         24                  1706201280                  0.0         0.0
1    ts-auth-service            trace.from_ts-auth-service.error_rate               24                         24                  1706201280                  0.0         0.0
2    ts-auth-service              trace.to_ts-auth-service.error_rate               24                         24                  1706201280                  0.0         0.0
3   ts-order-service                       trace.from_root.error_rate               24                         24                  1706201280                  0.0         0.0
4   ts-order-service  trace.from_ts-inside-payment-service.error_rate               24                         24                  1706201280                  0.0         0.0
5   ts-order-service           trace.from_ts-order-service.error_rate               24                         24                  1706201280                  0.0         0.0
6   ts-order-service        trace.from_ts-preserve-service.error_rate               24                         24                  1706201280                  0.0         0.0
7   ts-order-service            trace.from_ts-seat-service.error_rate               24                         24                  1706201280                  0.0         0.0
8   ts-order-service        trace.from_ts-security-service.error_rate               24                         24                  1706201280                  0.0         0.0
9   ts-order-service          trace.from_ts-travel-service.error_rate               24                         24                  1706201280                  0.0         0.0
10  ts-order-service             trace.to_ts-order-service.error_rate               24                         24                  1706201280                  0.0         0.0
11  ts-order-service           trace.to_ts-station-service.error_rate               24                         24                  1706201280                  0.0         0.0
12  ts-route-service           trace.from_ts-basic-service.error_rate               24                         24                  1706201280                  0.0         0.0
13  ts-route-service           trace.from_ts-route-service.error_rate               24                         24                  1706201280                  0.0         0.0
14  ts-route-service          trace.from_ts-travel-service.error_rate               24                         24                  1706201280                  0.0         0.0
15  ts-route-service         trace.from_ts-travel2-service.error_rate               24                         24                  1706201280                  0.0         0.0
16  ts-route-service             trace.to_ts-route-service.error_rate               24                         24                  1706201280                  0.0         0.0
17  ts-train-service           trace.from_ts-basic-service.error_rate               24                         24                  1706201280                  0.0         0.0
18  ts-train-service           trace.from_ts-train-service.error_rate               24                         24                  1706201280                  0.0         0.0
19  ts-train-service          trace.from_ts-travel-service.error_rate               24                         24                  1706201280                  0.0         0.0,                                           trace_name
0                      trace.from_root.duration_mean
1                       trace.from_root.duration_p95
2                         trace.from_root.error_rate
3                          trace.from_root.row_count
4   trace.from_ts-admin-travel-service.duration_mean
..                                               ...
45          trace.from_ts-train-service.duration_p95
46            trace.from_ts-train-service.error_rate
47             trace.from_ts-train-service.row_count
48        trace.from_ts-travel-service.duration_mean
49         trace.from_ts-travel-service.duration_p95

[50 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse loaded data if present, otherwise load from CSV
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
try:
    error_logs
except NameError:
    error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (rules)
for df in (log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure column names
if 'log_name' not in log.columns and 'log' in log.columns:
    log = log.rename(columns={'log': 'log_name'})

# Target services
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Incident window (UTC)
start = pd.to_datetime('2024-01-25 16:45:00', utc=True)
end = pd.to_datetime('2024-01-25 17:15:00', utc=True)

# -------- Part 1: log.csv analysis --------
# Restrict to target services
log_services = log[log['cmdb_id'].isin(services)].copy()

# Compute global P95 per (cmdb_id, log_name) using full series (no time filter)
global_p95_log = log_services.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95)
global_p95_log = global_p95_log.rename(columns={'value':'global_p95'})

# Filter logs to incident window
log_window = log_services[(log_services['timestamp'] >= start) & (log_services['timestamp'] <= end)].copy()

if log_window.empty:
    log_results = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ])
else:
    # Merge global p95 into windowed logs
    lw = log_window.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')
    # Flag anomalies: value >= global_p95
    lw['is_anomaly'] = lw['value'] >= lw['global_p95']
    # Aggregations
    grp = lw.groupby(['cmdb_id','log_name'], as_index=False)
    agg_counts = grp['value'].count().rename(columns={'value':'count_in_window'})
    agg_anoms = grp['is_anomaly'].sum().rename(columns={'is_anomaly':'count_anomalies_in_window'})
    agg_max = grp['value'].max().rename(columns={'value':'max_value_in_window'})
    agg_p95 = grp['global_p95'].first().rename(columns={'global_p95':'global_p95'})
    log_results = agg_counts.merge(agg_anoms, on=['cmdb_id','log_name'])
    log_results = log_results.merge(agg_max, on=['cmdb_id','log_name'])
    log_results = log_results.merge(agg_p95, on=['cmdb_id','log_name'])
    # Earliest anomaly timestamp per group (if any)
    anomalies = lw[lw['is_anomaly']]
    if not anomalies.empty:
        earliest = anomalies.groupby(['cmdb_id','log_name'], as_index=False)['timestamp'].min()
        earliest['earliest_anomaly_timestamp'] = earliest['timestamp'].apply(lambda ts: int(ts.timestamp()))
        earliest = earliest.drop(columns=['timestamp'])
        log_results = log_results.merge(earliest, on=['cmdb_id','log_name'], how='left')
    else:
        log_results['earliest_anomaly_timestamp'] = np.nan
    # Ensure types
    log_results['count_in_window'] = log_results['count_in_window'].astype(int)
    log_results['count_anomalies_in_window'] = log_results['count_anomalies_in_window'].astype(int)
    # Select and order columns
    log_results = log_results[[
        'cmdb_id','log_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ]]
# Sort and limit to top 20
log_results = log_results.sort_values(by=['count_anomalies_in_window','global_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# -------- Part 2: error_logs.csv analysis --------
# Restrict error logs to services and incident window
err = error_logs[error_logs['cmdb_id'].isin(services)].copy()
err_window = err[(err['timestamp'] >= start) & (err['timestamp'] <= end)].copy()

# Total number of error log entries per cmdb_id
error_counts_per_cmdb = err_window.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'error_count'})

# Top 20 error log rows sorted by timestamp ascending; include integer timestamp seconds
if err_window.empty:
    top_error_rows = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    top_err = err_window.sort_values('timestamp', ascending=True).head(20).copy()
    top_err['timestamp'] = top_err['timestamp'].apply(lambda ts: int(ts.timestamp()))
    top_error_rows = top_err[['timestamp','cmdb_id','message']].reset_index(drop=True)

# Keyword counts per cmdb_id (case-insensitive)
keywords = ["socket","timeout","connection","refused","broken","reset","packet","latency","IO","out of memory","OOM"]
# Prepare lower-case message for matching; handle NaN messages
err_window['message_lc'] = err_window['message'].fillna('').str.lower()

rows = []
for cmdb, group in err_window.groupby('cmdb_id'):
    for kw in keywords:
        # For case-insensitive match; match exact phrase lowercased
        kw_lc = kw.lower()
        # For 'io' and 'oom' keep the provided keyword as is (lowercase)
        # Use simple substring containment
        cnt = group['message_lc'].str.contains(kw_lc, regex=False).sum()
        if cnt > 0:
            rows.append({'cmdb_id': cmdb, 'keyword': kw, 'count': int(cnt)})

if rows:
    keyword_counts = pd.DataFrame(rows).sort_values(['cmdb_id','count'], ascending=[True, False]).reset_index(drop=True)
else:
    keyword_counts = pd.DataFrame(columns=['cmdb_id','keyword','count'])

# Final compact outputs
log_results, error_counts_per_cmdb, top_error_rows, keyword_counts

```
Out[4]:
```
summary = (
    "Summary of log and error_log analysis for 2024-01-25 16:45–17:15 UTC:\n\n"
    "1) log.csv findings:\n"
    "- log.error_count: For ts-auth-service, ts-order-service, ts-route-service, and ts-travel-service there are 24 points in the window each. "
    "global_p95 = 0.00 for these groups, and count_anomalies_in_window = 24 (every point met value >= global_p95). "
    "Earliest anomaly timestamp reported as 1706201280 for these groups.\n\n"
    "- log.row_count: Several services show 2 anomaly points each (value >= global_p95) within the window:\n"
    "  * ts-travel-service: count_in_window=24, count_anomalies_in_window=2, max_value_in_window=2227, global_p95=1890.25\n"
    "  * ts-route-service:  count_in_window=24, count_anomalies_in_window=2, max_value_in_window=1016, global_p95=1011.10\n"
    "  * ts-order-service:  count_in_window=24, count_anomalies_in_window=2, max_value_in_window=520,  global_p95=443.35\n"
    "  * ts-auth-service:   count_in_window=24, count_anomalies_in_window=2, max_value_in_window=332,  global_p95=330.20\n\n"
    "Note: log.error_count entries are flagged as anomalies because global_p95 is zero; this makes the anomaly flag trivial (0 >= 0). Consider treating zero-baseline metrics specially (e.g., require value > global_p95 or require global_p95 > 0) to avoid misleading alerts.\n\n"
    "2) error_logs.csv findings:\n"
    "- There are no error_log entries for the five services in the incident window (no rows returned). Consequently, per-cmdb_id error counts are empty and no keyword matches were found.\n\n"
    "Overall takeaway:\n"
    "- Some log.row_count values exceeded their historical P95 (notably ts-travel-service and ts-route-service).\n"
    "- Many log.error_count anomalies are artifacts of a zero global P95 baseline and should be interpreted with caution.\n"
    "- No textual error log evidence (error_logs.csv) was found in the window for these services."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp  max_value_in_window  global_p95
0    ts-auth-service  log.error_count               24                         24                  1706201280                    0        0.00
1   ts-order-service  log.error_count               24                         24                  1706201280                    0        0.00
2   ts-route-service  log.error_count               24                         24                  1706201280                    0        0.00
3  ts-travel-service  log.error_count               24                         24                  1706201280                    0        0.00
4  ts-travel-service    log.row_count               24                          2                  1706202180                 2227     1890.25
5   ts-route-service    log.row_count               24                          2                  1706201400                 1016     1011.10
6   ts-order-service    log.row_count               24                          2                  1706202180                  520      443.35
7    ts-auth-service    log.row_count               24                          2                  1706201460                  332      330.20, Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [cmdb_id, keyword, count]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse trace if available, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed and trace_name column exists
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
if 'trace_name' not in trace.columns and 'trace' in trace.columns:
    trace = trace.rename(columns={'trace':'trace_name'})

# Target services
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to target services
trace_services = trace[trace['cmdb_id'].isin(services)].copy()

# Further restrict trace_name to those containing "duration" (case-insensitive)
# OR containing "ts-route-service" (case-insensitive)
mask_duration = trace_services['trace_name'].str.contains('duration', case=False, na=False)
mask_route = trace_services['trace_name'].str.contains('ts-route-service', case=False, na=False)
trace_filtered = trace_services[mask_duration | mask_route].copy()

# Compute global P95 per (cmdb_id, trace_name) using full available series (no time filter)
global_p95_df = trace_filtered.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95)
global_p95_df = global_p95_df.rename(columns={'value': 'global_p95'})

# Incident window (UTC)
start = pd.to_datetime('2024-01-25 16:45:00', utc=True)
end = pd.to_datetime('2024-01-25 17:15:00', utc=True)

# Filter to the incident window
window_df = trace_filtered[(trace_filtered['timestamp'] >= start) & (trace_filtered['timestamp'] <= end)].copy()

if window_df.empty:
    final_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ])
else:
    # Merge global p95 into windowed data
    w = window_df.merge(global_p95_df, on=['cmdb_id','trace_name'], how='left')
    # Flag anomalies where value >= global_p95
    w['is_anomaly'] = w['value'] >= w['global_p95']
    # Aggregations
    grp = w.groupby(['cmdb_id','trace_name'], as_index=False)
    agg_counts = grp['value'].count().rename(columns={'value':'count_in_window'})
    agg_anoms = grp['is_anomaly'].sum().rename(columns={'is_anomaly':'count_anomalies_in_window'})
    agg_max = grp['value'].max().rename(columns={'value':'max_value_in_window'})
    agg_p95 = grp['global_p95'].first().rename(columns={'global_p95':'global_p95'})
    final_df = agg_counts.merge(agg_anoms, on=['cmdb_id','trace_name'])
    final_df = final_df.merge(agg_max, on=['cmdb_id','trace_name'])
    final_df = final_df.merge(agg_p95, on=['cmdb_id','trace_name'])
    # Earliest anomaly timestamp per group (if any)
    anomalies = w[w['is_anomaly']]
    if not anomalies.empty:
        earliest = anomalies.groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min()
        earliest['earliest_anomaly_timestamp'] = earliest['timestamp'].apply(lambda ts: int(ts.timestamp()))
        earliest = earliest.drop(columns=['timestamp'])
        final_df = final_df.merge(earliest, on=['cmdb_id','trace_name'], how='left')
    else:
        final_df['earliest_anomaly_timestamp'] = np.nan
    # Ensure integer counts
    final_df['count_in_window'] = final_df['count_in_window'].astype(int)
    final_df['count_anomalies_in_window'] = final_df['count_anomalies_in_window'].astype(int)
    # Order columns
    final_df = final_df[[
        'cmdb_id','trace_name','count_in_window','count_anomalies_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','global_p95'
    ]]

# Sort by count_anomalies_in_window desc then global_p95 desc, limit to 20
final_df = final_df.sort_values(by=['count_anomalies_in_window','global_p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# Distinct trace_name values found (limit 50)
distinct_trace_names = pd.DataFrame({'trace_name': sorted(trace_filtered['trace_name'].unique())[:50]})

# Display compact results
final_df, distinct_trace_names

```
Out[5]:
```
summary = (
    "Summary for traces (filtered to duration metrics and any trace involving ts-route-service) "
    "during 2024-01-25 16:45–17:15 UTC:\n\n"
    "- Distinct trace_name values found: 48 (limited list returned).\n\n"
    "- Findings:\n"
    "  * ts-route-service.trace.from_ts-route-service.error_rate and .to_ts-route-service.error_rate: "
    "24 points each in the window and flagged as anomalies, but global P95 = 0.0 and max = 0.0 — this is a zero-baseline artifact and should not be treated as a meaningful spike.\n"
    "  * ts-route-service row_count (from/to): 24 points, count_anomalies_in_window = 2, max_value = 2032, global_p95 = 2022.2, earliest anomaly at epoch 1706201400.\n"
    "  * ts-travel-service to_ts-route-service.row_count: 2 anomalies, max = 368, global_p95 = 321.4, earliest anomaly at 1706201400.\n"
    "  * Duration metrics across services (examples):\n"
    "      - ts-travel-service trace.from_root.duration_p95: 2 anomalies, max = 3.202661, global_p95 = 3.051135 (earliest 1706202240).\n"
    "      - ts-travel-service trace.from_ts-preserve-service.duration_p95: 2 anomalies, max = 2.317523, global_p95 = 1.820638 (earliest 1706202480).\n"
    "      - ts-auth-service trace.from_root.duration_p95: 2 anomalies, max = 1.228697, global_p95 = 0.897354 (earliest 1706201760).\n"
    "      - ts-order-service trace.from_root.duration_p95: 2 anomalies, max = 0.599286, global_p95 = 0.556094 (earliest 1706202360).\n\n"
    "- Interpretation and next steps:\n"
    "  * Ignore error_rate groups with global_p95 == 0.0 (they produce misleading anomaly counts). Use a stricter rule (e.g., value > global_p95 and global_p95 > 0) for those metrics.\n"
    "  * Investigate the ts-route-service row_count spikes (epoch 1706201400) and ts-travel-service row_count spike; these exceeded historical P95 by a noticeable margin.\n"
    "  * Review the duration increases for ts-travel-service (from_root and preserve-service) and the auth/order root durations — they show modest but consistent exceedances (2 points each) and may indicate latency regressions worth investigating.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                        trace_name  count_in_window  count_anomalies_in_window  earliest_anomaly_timestamp  max_value_in_window   global_p95
0    ts-route-service            trace.from_ts-route-service.error_rate               24                         24                  1706201280             0.000000     0.000000
1    ts-route-service              trace.to_ts-route-service.error_rate               24                         24                  1706201280             0.000000     0.000000
2   ts-travel-service              trace.to_ts-route-service.error_rate               24                         24                  1706201280             0.000000     0.000000
3    ts-route-service             trace.from_ts-route-service.row_count               24                          2                  1706201400          2032.000000  2022.200000
4    ts-route-service               trace.to_ts-route-service.row_count               24                          2                  1706201400          2032.000000  2022.200000
5   ts-travel-service               trace.to_ts-route-service.row_count               24                          2                  1706201400           368.000000   321.400000
6   ts-travel-service                      trace.from_root.duration_p95               24                          2                  1706202240             3.202661     3.051135
7   ts-travel-service       trace.from_ts-preserve-service.duration_p95               24                          2                  1706202480             2.317523     1.820638
8   ts-travel-service                     trace.from_root.duration_mean               24                          2                  1706202240             1.913400     1.798153
9   ts-travel-service      trace.from_ts-preserve-service.duration_mean               24                          2                  1706202300             2.317523     1.251633
10  ts-travel-service   trace.from_ts-admin-travel-service.duration_p95               24                          2                  1706202600             1.210054     1.197555
11    ts-auth-service                      trace.from_root.duration_p95               24                          2                  1706201760             1.228697     0.897354
12  ts-travel-service           trace.from_ts-food-service.duration_p95               24                          2                  1706202180             0.899949     0.856342
13   ts-order-service                      trace.from_root.duration_p95               24                          2                  1706202360             0.599286     0.556094
14   ts-order-service          trace.to_ts-station-service.duration_p95               24                          2                  1706202360             0.581904     0.538127
15   ts-order-service                     trace.from_root.duration_mean               24                          2                  1706202360             0.528466     0.526517
16   ts-order-service         trace.to_ts-station-service.duration_mean               24                          2                  1706202360             0.510086     0.507908
17  ts-travel-service  trace.from_ts-admin-travel-service.duration_mean               24                          2                  1706202420             0.498167     0.476397
18  ts-travel-service             trace.to_ts-seat-service.duration_p95               24                          2                  1706202060             0.777783     0.447581
19    ts-auth-service                     trace.from_root.duration_mean               24                          2                  1706202480             0.414664     0.364798,                                           trace_name
0                      trace.from_root.duration_mean
1                       trace.from_root.duration_p95
2   trace.from_ts-admin-travel-service.duration_mean
3    trace.from_ts-admin-travel-service.duration_p95
4           trace.from_ts-auth-service.duration_mean
..                                               ...
43       trace.to_ts-ticketinfo-service.duration_p95
44           trace.to_ts-train-service.duration_mean
45            trace.to_ts-train-service.duration_p95
46          trace.to_ts-travel-service.duration_mean
47           trace.to_ts-travel-service.duration_p95

[48 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```